In [34]:
import itertools
import math
import numpy as np
import random
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib qt

def calculate_distance(p1,p2):
    #can handle, lists, tuples, arrays
    return math.dist(p1,p2)

def get_path_endpoints(path, is_reversed):
    return (path[1], path[0]) if is_reversed else (path[0], path[1])

def calculate_sequence_cost(sequence, paths_data):
    total_cost=0
    for i in range(len(sequence)-1):
        current_path_idx, current_is_reversed = sequence[i]
        _, current_end_point= get_path_endpoints(paths_data[current_path_idx], current_is_reversed)
        next_path_idx, next_is_reversed= sequence[i+1]
        next_start_point, _ = get_path_endpoints(paths_data[next_path_idx], next_is_reversed)
        total_cost += calculate_distance(current_end_point, next_start_point)
    return total_cost


### Brute-Force all possible options

In [35]:
def find_optimal_path_order(paths):
    
    num_paths=len(paths)
    min_total_cost=float('inf')
    best_sequence= None

    #Generate all possible orderings of paths
    path_indices= range(num_paths)
    x=0
    for p_order in itertools.permutations(path_indices):
        #iterate through all possible orderings
        for direction in range(2**num_paths):  
            current_sequence=[]
            for i in range(num_paths):
                path_index= p_order[i]
                original_path= paths[path_index] # ordered list  [P_i_0, P_i_1]
                
                if (direction>>i) & 1: #compare i-th bit with 1, ensures all 32 orderings happen
                    current_sequence.append((original_path[1], original_path[0])) #Normal Direction
                else:
                    current_sequence.append((original_path[0], original_path[1])) #Flipped Direction
        
            #Calculate the "transition cost" for this specific sequence
            current_cost = 0
            for i in range(num_paths - 1):
                # Cost is the distance from the end of one path to the start of the next
                end_of_current_path = current_sequence[i][1]
                start_of_next_path = current_sequence[i+1][0]
                current_cost += calculate_distance(end_of_current_path, start_of_next_path)

            #If this sequence is the best so far, save it
            if current_cost < min_total_cost:
                min_total_cost = current_cost
                best_sequence = current_sequence
                        
    return min_total_cost, best_sequence
    
if __name__ == "__main__":
    # Define 5 sample paths with their terminal (start, end) 3D coordinates.
    # Each path is a tuple: ((x1, y1, z1), (x2, y2, z2))
    robot_paths = [
        [(0, 0, 0), (5, 5, 2)],      # Path 0
        [(20, 15, 8), (18, 12, 5)],  # Path 1
        [(7, 4, 3), (12, 10, 10)],   # Path 2
        [(25, 25, 10), (22, 20, 6)], # Path 3
        [(14, 9, 8), (19, 14, 9)]    # Path 4
    ]

    robot_paths1 = [
        ((0, 0, 0), (5, 5, 2)),      # Path 0
        ((20, 15, 8), (18, 12, 5)),  # Path 1
        ((7, 4, 3), (12, 10, 10)),   # Path 2
        ((25, 25, 10), (22, 20, 6)), # Path 3
        ((14, 9, 8), (19, 14, 9)),   # Path 4
        ((40, 40, 40), (45, 42, 38)),# Path 5
        ((5, 25, 15), (8, 28, 12)),  # Path 6
        ((30, 10, 5), (35, 12, 2))   # Path 7
    ]

    # Run the optimization function
    min_cost, optimal_sequence = find_optimal_path_order(robot_paths1)

    # Print the results
    print("Optimization Complete!")
    print("-------------------------")
    print(f"Minimum Connection Cost: {min_cost:.4f}\n")
    print("Optimal Path Sequence (Start -> End):")
    for i, path in enumerate(optimal_sequence):
        print(f"  Step {i+1}:  {path[0]} -> {path[1]}")



Optimization Complete!
-------------------------
Minimum Connection Cost: 98.0571

Optimal Path Sequence (Start -> End):
  Step 1:  (0, 0, 0) -> (5, 5, 2)
  Step 2:  (7, 4, 3) -> (12, 10, 10)
  Step 3:  (14, 9, 8) -> (19, 14, 9)
  Step 4:  (20, 15, 8) -> (18, 12, 5)
  Step 5:  (30, 10, 5) -> (35, 12, 2)
  Step 6:  (22, 20, 6) -> (25, 25, 10)
  Step 7:  (8, 28, 12) -> (5, 25, 15)
  Step 8:  (40, 40, 40) -> (45, 42, 38)


### Nearest Neighbor + 2-Opt Algorithm




In [36]:
def adapted_nearest_neighbor(start_node, paths_data):
    
    num_paths=len(paths_data)
    paths_to_visit = set(range(num_paths)) #for set for efficient checking

    start_path_idx, start_is_reversed = start_node

    sequence = [start_node]
    paths_to_visit.remove(start_path_idx)

    while paths_to_visit:
        last_path_idx, last_is_reversed = sequence[-1]
        _, last_end_point= get_path_endpoints(paths_data[last_path_idx], last_is_reversed)
        
        best_dist= float("inf")
        best_next_path_idx= -1
        best_next_is_reversed=False

        #find nearest terminal point (could be start/end of path)
        for next_path_idx in paths_to_visit:
            p_start,_= get_path_endpoints(paths_data[next_path_idx], False)
            dist= calculate_distance(last_end_point, p_start)
            if dist < best_dist:
                best_dist= dist
                best_next_path_idx= next_path_idx
                best_next_is_reversed= False

            p_start_reverse, _ = get_path_endpoints(paths_data[next_path_idx], True)
            dist_reverse = calculate_distance(last_end_point, p_start_reverse)
            if dist_reverse < best_dist:
                best_dist = dist_reverse
                best_next_path_idx = next_path_idx
                best_next_is_reversed= True

        sequence.append((best_next_path_idx, best_next_is_reversed))
        paths_to_visit.remove(best_next_path_idx)
    
    return sequence


def adapted_two_opt(sequence, paths_data):
    best_sequence = sequence
    improved = True
    while improved:
        improved = False
        best_cost = calculate_sequence_cost(best_sequence, paths_data)

        for i in range(1, len(best_sequence)-1):
            for j in range (i+1, len(best_sequence)):
                new_sequence = best_sequence[:i] + best_sequence[i: j+1][::-1] + best_sequence[j+1:]
                new_cost = calculate_sequence_cost(new_sequence, paths_data)

                if new_cost < best_cost:
                    print("swap")
                    best_sequence = new_sequence
                    best_cost = new_cost
                    improved = True



    return best_sequence

def find_best_starting_point(paths_data):

    #find the point furthest from the centroid
    all_points=[]
    for i, path in enumerate(paths_data):
        all_points.append({"coord": path[0], 'path_idx': i, "is_reversed": False})
        all_points.append({"coord": path[1], 'path_idx': i, "is_reversed": True})

    sum_x, sum_y, sum_z=0,0,0
    for point in all_points:
        sum_x += point['coord'][0]
        sum_y += point['coord'][1]
        sum_z += point["coord"][2]
    n=len(all_points)
    centroid = (sum_x/n, sum_y/n, sum_z/n)

    max_dist = -1
    best_start = None
    for point in all_points:
        dist = calculate_distance(point["coord"], centroid)
        if dist> max_dist:
            max_dist=dist
            best_start= (point["path_idx"], point["is_reversed"])

    return best_start



if __name__ == "__main__":
    # Define 5 sample paths with their terminal (start, end) 3D coordinates.
    # Each path is a tuple: ((x1, y1, z1), (x2, y2, z2))
    robot_paths = [
        [(0, 0, 0), (5, 5, 2)],      # Path 0
        [(20, 15, 8), (18, 12, 5)],  # Path 1
        [(7, 4, 3), (12, 10, 10)],   # Path 2
        [(25, 25, 10), (22, 20, 6)], # Path 3
        [(14, 9, 8), (19, 14, 9)]    # Path 4
    ]

    robot_paths1 = [
        ((0, 0, 0), (5, 5, 2)),      # Path 0
        ((20, 15, 8), (18, 12, 5)),  # Path 1
        ((7, 4, 3), (12, 10, 10)),   # Path 2
        ((25, 25, 10), (22, 20, 6)), # Path 3
        ((14, 9, 8), (19, 14, 9)),   # Path 4
        ((40, 40, 40), (45, 42, 38)),# Path 5
        ((5, 25, 15), (8, 28, 12)),  # Path 6
        ((30, 10, 5), (35, 12, 2))   # Path 7
    ]

    paths = robot_paths1
    best_start_node = find_best_starting_point(paths)
    print(f"Best start point: {best_start_node[0]}")

    initial_sequence= adapted_nearest_neighbor(best_start_node, paths)
    initial_cost= calculate_sequence_cost(initial_sequence, paths)
    print("--- Nearest Neighbor Solution (Initial Guess) ---")
    print(f"Connection Cost: {initial_cost:.4f} \n")

    optimized_sequence = adapted_two_opt(initial_sequence, paths)
    optimized_cost = calculate_sequence_cost(optimized_sequence, paths)

    print("--- 2-Opt Solution (Improved) ---")
    print(f"Final Connection Cost: {optimized_cost:.4f}\n")
    print("Final Path Sequence (Start -> End):")
    
    final_path_coords = []
    for path_idx, is_reversed in optimized_sequence:
        start_point, end_point = get_path_endpoints(paths[path_idx], is_reversed)
        final_path_coords.append((start_point, end_point))

    for i, (start_p, end_p) in enumerate(final_path_coords):
        print(f"  Step {i+1}:  {start_p} -> {end_p}")




Best start point: 5
--- Nearest Neighbor Solution (Initial Guess) ---
Connection Cost: 111.3739 

--- 2-Opt Solution (Improved) ---
Final Connection Cost: 111.3739

Final Path Sequence (Start -> End):
  Step 1:  (45, 42, 38) -> (40, 40, 40)
  Step 2:  (25, 25, 10) -> (22, 20, 6)
  Step 3:  (20, 15, 8) -> (18, 12, 5)
  Step 4:  (19, 14, 9) -> (14, 9, 8)
  Step 5:  (12, 10, 10) -> (7, 4, 3)
  Step 6:  (5, 5, 2) -> (0, 0, 0)
  Step 7:  (5, 25, 15) -> (8, 28, 12)
  Step 8:  (30, 10, 5) -> (35, 12, 2)


### Simulated Annealing (Metaheuristic)

In [50]:
def get_random_neighbor_sequence(sequence):
    new_sequence = list(sequence)
    #either swaps random paths or flips start/end order of 1 pass
    if random.random() <0.5:
        i,j=random.sample(range(len(new_sequence)), 2)
        new_sequence[i], new_sequence[j]= new_sequence[j], new_sequence[i]
    else:
        i=random.randrange(len(new_sequence))
        path_idx, is_reversed= new_sequence[i]  
        new_sequence[i] = (path_idx, not is_reversed)
    return new_sequence

def simulated_annealing(paths_data, start_temp, end_temp, cooling_rate):
    num_paths = len(paths_data)

    current_sequence=[]
    path_indices = list(range(num_paths))
    random.shuffle(path_indices)
    for idx in path_indices:
        current_sequence.append((idx, random.choice([True,False])))

    current_cost = calculate_sequence_cost(current_sequence, paths_data)
    best_sequence= current_sequence
    best_cost = current_cost
    temperature=start_temp

    while temperature>end_temp:
        neighbor_sequence = get_random_neighbor_sequence(current_sequence)
        neighbor_cost = calculate_sequence_cost(neighbor_sequence, paths_data)

        cost_difference = neighbor_cost - current_cost
        if cost_difference <0:
            current_sequence = neighbor_sequence
            current_cost = neighbor_cost
        else: #Accept swap even if worse to prevent local minima (more likely if close to current best)
            acceptance_probability = math.exp(-cost_difference/temperature)
            if random.random() < acceptance_probability:
                current_sequence=neighbor_sequence
                current_cost = neighbor_cost

        if current_cost < best_cost:
            best_sequence = current_sequence
            best_cost = current_cost
        
        temperature *=cooling_rate

    return best_cost, best_sequence

if __name__ == "__main__":
    robot_paths = [
        ((0, 0, 0), (5, 5, 2)),      # Path 0
        ((20, 15, 8), (18, 12, 5)),  # Path 1
        ((7, 4, 3), (12, 10, 10)),   # Path 2
        ((25, 25, 10), (22, 20, 6)), # Path 3
        ((14, 9, 8), (19, 14, 9))    # Path 4
    ]

    robot_paths1 = [
        ((0, 0, 0), (5, 5, 2)),      # Path 0
        ((20, 15, 8), (18, 12, 5)),  # Path 1
        ((7, 4, 3), (12, 10, 10)),   # Path 2
        ((25, 25, 10), (22, 20, 6)), # Path 3
        ((14, 9, 8), (19, 14, 9)),   # Path 4
        ((40, 40, 40), (45, 42, 38)),# Path 5
        ((5, 25, 15), (8, 28, 12)),  # Path 6
        ((30, 10, 5), (35, 12, 2))   # Path 7
    ]

    # --- Hyperparameters for the algorithm ---
    STARTING_TEMPERATURE = 1000   # High enough to accept many bad moves
    ENDING_TEMPERATURE = 1        # Low enough to be "frozen"
    COOLING_RATE = 0.999          # Slow cooling (e.g., 0.999) is better
    
    paths=robot_paths1
    final_cost, final_sequence = simulated_annealing(
        paths, STARTING_TEMPERATURE, ENDING_TEMPERATURE, COOLING_RATE
    )

    print("--- Simulated Annealing Solution ---")
    print(f"Final Connection Cost: {final_cost:.4f}\n")
    print("Final Path Sequence (Start -> End):")
    
    final_annealed_coords = []
    for path_idx, is_reversed in final_sequence:
        start_point, end_point = get_path_endpoints(paths[path_idx], is_reversed)
        final_annealed_coords.append((start_point, end_point))

    for i, (start_p, end_p) in enumerate(final_path_coords):
        print(f"  Step {i+1}:  {start_p} -> {end_p}")


--- Simulated Annealing Solution ---
Final Connection Cost: 98.0571

Final Path Sequence (Start -> End):
  Step 1:  (45, 42, 38) -> (40, 40, 40)
  Step 2:  (25, 25, 10) -> (22, 20, 6)
  Step 3:  (20, 15, 8) -> (18, 12, 5)
  Step 4:  (19, 14, 9) -> (14, 9, 8)
  Step 5:  (12, 10, 10) -> (7, 4, 3)
  Step 6:  (5, 5, 2) -> (0, 0, 0)
  Step 7:  (5, 25, 15) -> (8, 28, 12)
  Step 8:  (30, 10, 5) -> (35, 12, 2)


## Paths Visualization

In [51]:
def plot_paths_3d(original_paths, optimal_sequence):
    fig=plt.figure(figsize=(10,8))
    ax = fig.add_subplot(111, projection='3d')

    # Plot sequences
    for i, (p_start,p_end) in enumerate(original_paths):
        ax.plot([p_start[0], p_end[0]], [p_start[1], p_end[1]],[p_start[2], p_end[2]],
                'o--',)
        
    # Plot optimal sequence connections
    if optimal_sequence:
        # Extract all points in the optimal sequence
        all_points_x = []
        all_points_y = []
        all_points_z = []

        print("\nPlotting Optimal Sequence:")
        for i, (start_node, end_node) in enumerate(optimal_sequence):
            print(f"  Path {i+1}: {start_node} -> {end_node}")
            all_points_x.extend([start_node[0], end_node[0]])
            all_points_y.extend([start_node[1], end_node[1]])
            all_points_z.extend([start_node[2], end_node[2]])
            
            # Plot the path segment itself in the optimal order
            ax.plot([start_node[0], end_node[0]], [start_node[1], end_node[1]], [start_node[2], end_node[2]],
                    'b-', linewidth=3, alpha=0.8) # Blue solid line for the path itself

            # Plot the connection line to the next path
            if i < len(optimal_sequence) - 1:
                next_start_node = optimal_sequence[i+1][0]
                ax.plot([end_node[0], next_start_node[0]], 
                        [end_node[1], next_start_node[1]], 
                        [end_node[2], next_start_node[2]],
                        'r:', linewidth=2, alpha=0.7) # Red dotted line for transition

        # Mark the start and end of the overall robot path with larger markers
        ax.plot([optimal_sequence[0][0][0]], [optimal_sequence[0][0][1]], [optimal_sequence[0][0][2]],
                'go', markersize=10, label='Overall Start')
        ax.plot([optimal_sequence[-1][1][0]], [optimal_sequence[-1][1][1]], [optimal_sequence[-1][1][2]],
                'ro', markersize=10, label='Overall End')
        ax.set_aspect('equal')


    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    ax.set_zlabel('Z Coordinate')
    ax.set_title('3D Robot Paths and Optimal Connections')
    ax.legend()
    ax.grid(True)
    plt.show()


plot_paths_3d(robot_paths, optimal_sequence)
plot_paths_3d(robot_paths, final_path_coords)
plot_paths_3d(robot_paths, final_annealed_coords)



Plotting Optimal Sequence:
  Path 1: (0, 0, 0) -> (5, 5, 2)
  Path 2: (7, 4, 3) -> (12, 10, 10)
  Path 3: (14, 9, 8) -> (19, 14, 9)
  Path 4: (20, 15, 8) -> (18, 12, 5)
  Path 5: (30, 10, 5) -> (35, 12, 2)
  Path 6: (22, 20, 6) -> (25, 25, 10)
  Path 7: (8, 28, 12) -> (5, 25, 15)
  Path 8: (40, 40, 40) -> (45, 42, 38)

Plotting Optimal Sequence:
  Path 1: (45, 42, 38) -> (40, 40, 40)
  Path 2: (25, 25, 10) -> (22, 20, 6)
  Path 3: (20, 15, 8) -> (18, 12, 5)
  Path 4: (19, 14, 9) -> (14, 9, 8)
  Path 5: (12, 10, 10) -> (7, 4, 3)
  Path 6: (5, 5, 2) -> (0, 0, 0)
  Path 7: (5, 25, 15) -> (8, 28, 12)
  Path 8: (30, 10, 5) -> (35, 12, 2)

Plotting Optimal Sequence:
  Path 1: (0, 0, 0) -> (5, 5, 2)
  Path 2: (7, 4, 3) -> (12, 10, 10)
  Path 3: (14, 9, 8) -> (19, 14, 9)
  Path 4: (20, 15, 8) -> (18, 12, 5)
  Path 5: (30, 10, 5) -> (35, 12, 2)
  Path 6: (22, 20, 6) -> (25, 25, 10)
  Path 7: (8, 28, 12) -> (5, 25, 15)
  Path 8: (40, 40, 40) -> (45, 42, 38)
